# Course 03: Human Approval & Permissions

In this module, we explore how to design a deterministic, secure human-in-the-loop (HITL) subsystem for highly consequential agentic operations.

We will learn that:
1. **Authentication != Authorization != Approval != Execution.**
2. The Orchestration layer (e.g., LangGraph) provides durable pause/resume, but is **NOT** the security boundary.
3. Human decisions must be cryptographically bound to the exact payload digest.
4. Applications must compute risk and idempotency keys deterministically.

## 1. Imports & Setup

In [ ]:
import sys
import os
sys.path.append(os.path.abspath("curriculum/intermediate/03-human-approval-permissions"))
from datetime import datetime, timezone, timedelta
import hashlib

from policy import (
    Service, Region, Environment, RiskTier, DecisionType, EventType, ExecutionStatus,
    RollbackProposal, ExecutionContext, ReviewerContext, EvidenceRef,
    build_approval_payload, ApprovalDecision, validate_approval, process_decision, ApprovalStore, PolicyError
)

store = ApprovalStore()

## 2. Agent Proposes Action
The AI Agent investigates an alert, determines there is a critical checkout failure, and generates a business-intent proposal.

In [ ]:
# The model generates the core intent.
proposal = RollbackProposal(
    service=Service.CHECKOUT,
    region=Region.EU_WEST,
    deployment_id="deploy-1842",
    reason="Critical conversion drop detected in checkout funnel."
)

# The Application binds the evidence that justified the proposal
evidence_refs = [
    EvidenceRef(
        evidence_id="alert-992",
        source_version="v1.0.3",
        observed_at=datetime.now(timezone.utc),
        max_age_seconds=3600
    )
]

# The Application provides its trusted context
execution_context = ExecutionContext(
    tenant_id="acme_corp",
    environment=Environment.PRODUCTION,
    request_id="req_8842",
    policy_version="v1"
)

# The Application deterministically calculates risk and TTL to build the final Payload
payload = build_approval_payload(proposal, execution_context, evidence_refs)

print(f"Risk Tier computed as: {payload.risk_tier.value}")
print(f"Payload bound by digest: {payload.display_digest}")

## 3. Human Review
The Orchestration framework pauses (via LangGraph `interrupt`) and the user reviews the request on a dashboard. They supply a decision.

In [ ]:
reviewer = ReviewerContext(
    reviewer_id="bob_the_sre",
    tenant_id="acme_corp",
    roles={"incident_commander", "sre_lead"},
    authenticated=True
)

decision = ApprovalDecision(
    decision=DecisionType.APPROVE,
    approver_id="bob_the_sre",
    approved_digest=payload.digest,
    reason="Rollback authorized to mitigate checkout failure.",
    decided_at=datetime.now(timezone.utc)
)

# Note: process_decision routes the state machine and yields a RollbackCommand on success.
try:
    current_evidence_state = {ev.evidence_id: ev for ev in evidence_refs}
    command = process_decision(store, "run-abc", payload, [decision], [reviewer], proposer_id="agent_1", current_policy_version="v1", current_evidence_state=current_evidence_state)
    print("\nDecision processed successfully. Command generated:")
    print(f"Idempotency Key: {command.idempotency_key}")
    print(f"Action Digest: {command.action_digest}")
except PolicyError as e:
    print(f"Authorization Denied: {e.code}")

## 4. Idempotent Execution
The execution engine guarantees exact-once processing semantics using the action_digest.

In [ ]:
# Attempt First Execution
receipt_1 = store.check_idempotency(command)
if not receipt_1:
    print("Executing Rollback...")
    receipt_1 = store.record_execution(command, ExecutionStatus.EXECUTED)
print(f"First Try: {receipt_1.status.value}")

# Simulate Network Timeout & Framework Retry
print("\nSimulating retry...")
receipt_2 = store.check_idempotency(command)
if receipt_2:
    print(f"Second Try: {receipt_2.status.value} - Skipping Execution!")

## 5. Audit Lifecycle
The Application Store tracks every transition deterministically.

In [ ]:
for event in store.audit_events:
    print(f"[{event.timestamp.isoformat()}] {event.event_type.value}: {event.reason}")

## 6. Evaluation Harness: Failure Mode Validation
Below, we test the rigorous security constraints encoded in our deterministc policy engine.

In [ ]:
def run_eval():
    failures = 0
    passed = 0
    
    def assert_failure(test_name, payload, decisions, reviewers, proposer, version, expected_error):
        nonlocal failures, passed
        try:
            current_evidence_state = {ev.evidence_id: ev for ev in payload.evidence_refs}
            process_decision(store, "eval", payload, decisions, reviewers, proposer, version, current_evidence_state)
            print(f"❌ {test_name}: Expected {expected_error}, but succeeded.")
            failures += 1
        except PolicyError as e:
            if e.code == expected_error:
                print(f"✅ {test_name}: correctly caught {expected_error}")
                passed += 1
            else:
                print(f"❌ {test_name}: Expected {expected_error}, got {e.code}")
                failures += 1

    print("--- Running Evaluation Harness ---")
    
    # 1. Digest Mutation (Reviewer tries to approve a mutated payload)
    d_mut = decision.model_copy(update={"approved_digest": "tampered_digest"})
    assert_failure("Digest Mutation", payload, [d_mut], [reviewer], "agent", "v1", "DIGEST_MISMATCH")
    
    # 2. Reviewer ID Mismatch
    d_id = decision.model_copy(update={"approver_id": "mallory"})
    assert_failure("Approver ID Mismatch", payload, [d_id], [reviewer], "agent", "v1", "REVIEWER_ID_MISMATCH")
    
    # 3. Context Length Mismatch
    assert_failure("Lengths Mismatch", payload, [decision, decision], [reviewer], "agent", "v1", "APPROVAL_CONTEXT_MISMATCH")
    
    # 4. Wrong Tenant
    r_tenant = reviewer.model_copy(update={"tenant_id": "evil_corp"})
    assert_failure("Cross-Tenant Access", payload, [decision], [r_tenant], "agent", "v1", "WRONG_TENANT")
    
    # 5. Unauthorized Role
    r_role = reviewer.model_copy(update={"roles": {"operator"}})
    assert_failure("Unauthorized Role", payload, [decision], [r_role], "agent", "v1", "UNAUTHORIZED_REVIEWER")
    
    # 6. Unauthenticated
    r_auth = reviewer.model_copy(update={"authenticated": False})
    assert_failure("Unauthenticated", payload, [decision], [r_auth], "agent", "v1", "UNAUTHENTICATED_REVIEWER")
    
    # 7. Expired TTL
    p_exp = payload.model_copy(update={"expires_at": datetime.now(timezone.utc) - timedelta(hours=1)})
    assert_failure("Expired Approval", p_exp, [decision], [reviewer], "agent", "v1", "EXPIRED_APPROVAL")
    
    # 8. Stale Evidence
    p_stale = payload.model_copy(update={"evidence_refs": [EvidenceRef(evidence_id="x", source_version="1", observed_at=datetime.now(timezone.utc) - timedelta(hours=2), max_age_seconds=3600)]})
    assert_failure("Stale Evidence", p_stale, [decision], [reviewer], "agent", "v1", "STALE_EVIDENCE")
    
    # 9. Risk Downgrade Attack
    p_risk = payload.model_copy(update={"risk_tier": RiskTier.LOW})
    assert_failure("Risk Recomputation Mismatch", p_risk, [decision], [reviewer], "agent", "v1", "RISK_MISMATCH")
    
    # 10. Separation of Duties (Proposer is Approver)
    assert_failure("SoD (Proposer==Approver)", payload, [decision], [reviewer], "bob_the_sre", "v1", "SEPARATION_OF_DUTIES_VIOLATION")
    
    # 11. Policy Version Change
    assert_failure("Policy Version Mismatch", payload, [decision], [reviewer], "agent", "v2", "POLICY_CHANGED")
    
    # 12. Critical Two-Person Coverage (Need incident_commander AND sre_lead)
    p_crit = build_approval_payload(RollbackProposal(service=Service.CHECKOUT, region=Region.GLOBAL, deployment_id="d2", reason=""), execution_context, [])
    r_ic1 = reviewer.model_copy(update={"reviewer_id": "ic1", "roles": {"incident_commander"}})
    r_ic2 = reviewer.model_copy(update={"reviewer_id": "ic2", "roles": {"incident_commander"}})
    d_ic1 = ApprovalDecision(decision=DecisionType.APPROVE, approver_id="ic1", approved_digest=p_crit.digest, reason="ok", decided_at=datetime.now(timezone.utc))
    d_ic2 = ApprovalDecision(decision=DecisionType.APPROVE, approver_id="ic2", approved_digest=p_crit.digest, reason="ok", decided_at=datetime.now(timezone.utc))
    
    assert_failure("Missing Required Role (Two-Person)", p_crit, [d_ic1, d_ic2], [r_ic1, r_ic2], "agent", "v1", "MISSING_REQUIRED_REVIEWER_ROLE")
    
    print(f"\nEval Summary: {passed} passed, {failures} failed.")
    assert failures == 0, "Evaluation failures detected."

run_eval()